In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1)



In [2]:
conv1 = nn.Conv2d(3, 16, 3, padding=1)
conv2 = nn.Conv2d(16, 32, 3, padding=1)
conv3 = nn.Conv2d(32, 16, 3, padding=1)
conv4 = nn.Conv2d(16, 1, 1)

x = torch.randn(2, 3, 64, 64)
target = torch.randint(0, 2, (2, 1, 64, 64)).float()

In [3]:
a1 = torch.relu(conv1(x))
a2 = torch.relu(conv2(F.max_pool2d(a1, 2)))

In [4]:
a3 = F.interpolate(a2, size=a1.shape[-2:], mode="bilinear",
                   align_corners=False)


In [5]:
a3 = a3[:, :16, :, :] + a1

output = conv4(torch.relu(a3))

In [6]:
prob = torch.sigmoid(output)

intersection = (prob * target).sum()

dice = (2*intersection + 1e-6) / (
    prob.sum() + target.sum() + 1e-6
)
dice_loss = 1 - dice

In [7]:
bce_loss = F.binary_cross_entropy_with_logits(output, target)

total_loss = dice_loss + bce_loss

print("Dice Loss:", dice_loss.item())
print("BCE Loss:", bce_loss.item())
print("Total Loss:", total_loss.item())

Dice Loss: 0.45802128314971924
BCE Loss: 0.7042173147201538
Total Loss: 1.162238597869873
